# Data Preprocessing

## Load Libraries

In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.preprocessing import MinMaxScaler

## Define columns

In [2]:
columns = ['engine_id', 'cycle', 'setting1', 'setting2', 'setting3']

for i in range(1, 22):
    columns.append(f'sensor{i}')

print(len(columns))  
print(columns)

26
['engine_id', 'cycle', 'setting1', 'setting2', 'setting3', 'sensor1', 'sensor2', 'sensor3', 'sensor4', 'sensor5', 'sensor6', 'sensor7', 'sensor8', 'sensor9', 'sensor10', 'sensor11', 'sensor12', 'sensor13', 'sensor14', 'sensor15', 'sensor16', 'sensor17', 'sensor18', 'sensor19', 'sensor20', 'sensor21']


## Load the dataset

In [3]:
train = pd.read_csv(
    "train_FD003.txt",
    sep=r"\s+",
    header=None
)

test = pd.read_csv(
    "test_FD003.txt",
    sep=r"\s+",
    header=None
)

rul = pd.read_csv(
    "RUL_FD003.txt",
    header=None
)

In [4]:
train.columns = columns
test.columns = columns

In [5]:
print(train.columns)

Index(['engine_id', 'cycle', 'setting1', 'setting2', 'setting3', 'sensor1',
       'sensor2', 'sensor3', 'sensor4', 'sensor5', 'sensor6', 'sensor7',
       'sensor8', 'sensor9', 'sensor10', 'sensor11', 'sensor12', 'sensor13',
       'sensor14', 'sensor15', 'sensor16', 'sensor17', 'sensor18', 'sensor19',
       'sensor20', 'sensor21'],
      dtype='str')


In [6]:
train.head()

,engine_id,cycle,setting1,setting2,setting3,sensor1,sensor2,sensor3,sensor4,sensor5,...,sensor12,sensor13,sensor14,sensor15,sensor16,sensor17,sensor18,sensor19,sensor20,sensor21
0,1,1,-0.0005,0.0004,100.0,518.67,642.36,1583.23,1396.84,14.62,...,522.31,2388.01,8145.32,8.4246,0.03,391,2388,100.0,39.11,23.3537
1,1,2,0.0008,-0.0003,100.0,518.67,642.50,1584.69,1396.89,14.62,...,522.42,2388.03,8152.85,8.4403,0.03,392,2388,100.0,38.99,23.4491
2,1,3,-0.0014,-0.0002,100.0,518.67,642.18,1582.35,1405.61,14.62,...,522.03,2388.00,8150.17,8.3901,0.03,391,2388,100.0,38.85,23.3669
3,1,4,-0.0020,0.0001,100.0,518.67,642.92,1585.61,1392.27,14.62,...,522.49,2388.08,8146.56,8.3878,0.03,392,2388,100.0,38.96,23.2951
4,1,5,0.0016,0.0000,100.0,518.67,641.68,1588.63,1397.65,14.62,...,522.58,2388.03,8147.80,8.3869,0.03,392,2388,100.0,39.14,23.4583


In [7]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 24720 entries, 0 to 24719
Data columns (total 26 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   engine_id  24720 non-null  int64  
 1   cycle      24720 non-null  int64  
 2   setting1   24720 non-null  float64
 3   setting2   24720 non-null  float64
 4   setting3   24720 non-null  float64
 5   sensor1    24720 non-null  float64
 6   sensor2    24720 non-null  float64
 7   sensor3    24720 non-null  float64
 8   sensor4    24720 non-null  float64
 9   sensor5    24720 non-null  float64
 10  sensor6    24720 non-null  float64
 11  sensor7    24720 non-null  float64
 12  sensor8    24720 non-null  float64
 13  sensor9    24720 non-null  float64
 14  sensor10   24720 non-null  float64
 15  sensor11   24720 non-null  float64
 16  sensor12   24720 non-null  float64
 17  sensor13   24720 non-null  float64
 18  sensor14   24720 non-null  float64
 19  sensor15   24720 non-null  float64
 20  sensor16   24720 

In [8]:
print(train.isnull().sum())

print(test.isnull().sum())

engine_id    0
cycle        0
setting1     0
setting2     0
setting3     0
sensor1      0
sensor2      0
sensor3      0
sensor4      0
sensor5      0
sensor6      0
sensor7      0
sensor8      0
sensor9      0
sensor10     0
sensor11     0
sensor12     0
sensor13     0
sensor14     0
sensor15     0
sensor16     0
sensor17     0
sensor18     0
sensor19     0
sensor20     0
sensor21     0
dtype: int64
engine_id    0
cycle        0
setting1     0
setting2     0
setting3     0
sensor1      0
sensor2      0
sensor3      0
sensor4      0
sensor5      0
sensor6      0
sensor7      0
sensor8      0
sensor9      0
sensor10     0
sensor11     0
sensor12     0
sensor13     0
sensor14     0
sensor15     0
sensor16     0
sensor17     0
sensor18     0
sensor19     0
sensor20     0
sensor21     0
dtype: int64


In [9]:
print(train.duplicated().sum())
print(test.duplicated().sum())

0
0


In [10]:
# Find the maximum cycle for each engine
max_cycle = train.groupby('engine_id')['cycle'].max().reset_index()

max_cycle.columns = ['engine_id', 'max_cycle']

# Merge with original data
train = train.merge(max_cycle, on='engine_id')

# Calculate Remaining Useful Life
train['RUL'] = train['max_cycle'] - train['cycle']

# Remove temporary column
train.drop('max_cycle', axis=1, inplace=True)

In [11]:
train[['engine_id', 'cycle', 'RUL']].head(10)

,engine_id,cycle,RUL
0,1,1,258
1,1,2,257
2,1,3,256
3,1,4,255
4,1,5,254
5,1,6,253
6,1,7,252
7,1,8,251
8,1,9,250
9,1,10,249


## Finding columns with constant features

In [12]:
train.nunique().sort_values()

sensor1         1
setting3        1
sensor5         1
sensor18        1
sensor19        1
sensor16        1
sensor10        4
sensor17       12
setting2       14
sensor6        17
engine_id     100
setting1      160
sensor8       161
sensor13      163
sensor20      165
sensor11      170
sensor2       334
cycle         525
RUL           525
sensor12     1772
sensor7      1854
sensor15     3122
sensor3      3358
sensor4      4383
sensor14     6320
sensor21     6440
sensor9      7114
dtype: int64

In [13]:
drop_cols = [
    'setting3',
    'sensor1',
    'sensor5',
    'sensor16',
    'sensor18',
    'sensor19'
]

train.drop(columns=drop_cols, inplace=True)
test.drop(columns=drop_cols, inplace=True)

## Normalization

In [14]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

In [15]:
feature_cols = train.columns.difference(['engine_id', 'cycle', 'RUL'])

In [16]:
train[feature_cols] = scaler.fit_transform(train[feature_cols])

In [17]:
test[feature_cols] = scaler.transform(test[feature_cols])

In [18]:
train.head()

,engine_id,cycle,setting1,setting2,sensor2,sensor3,sensor4,sensor6,sensor7,sensor8,...,sensor10,sensor11,sensor12,sensor13,sensor14,sensor15,sensor17,sensor20,sensor21,RUL
0,1,1,0.470930,0.769231,0.355972,0.370523,0.308580,1.0,0.208812,0.623529,...,0.333333,0.348571,0.231279,0.642857,0.239116,0.647755,0.272727,0.559524,0.446331,258
1,1,2,0.546512,0.230769,0.388759,0.399100,0.309360,1.0,0.236590,0.647059,...,0.333333,0.308571,0.236882,0.654762,0.278567,0.685659,0.363636,0.488095,0.534836,257
2,1,3,0.418605,0.307692,0.313817,0.353298,0.445398,1.0,0.230843,0.664706,...,0.333333,0.302857,0.217015,0.636905,0.264526,0.564462,0.272727,0.404762,0.458577,256
3,1,4,0.383721,0.538462,0.487119,0.417107,0.237285,1.0,0.268199,0.647059,...,0.333333,0.314286,0.240448,0.684524,0.245612,0.558909,0.363636,0.470238,0.391966,255
4,1,5,0.593023,0.461538,0.196721,0.476218,0.321217,1.0,0.245690,0.670588,...,0.333333,0.262857,0.245033,0.654762,0.252109,0.556736,0.363636,0.577381,0.543371,254


## Capping RUL column

### capping lets the model focus more on degradation and less on the healthy region

In [19]:
train['RUL'] = train['RUL'].clip(upper=125)

In [20]:
print(train['RUL'].max())

125


## Save processed dataset

In [21]:
train.to_csv("processed_train.csv", index=False)
test.to_csv("processed_test.csv", index=False)

In [24]:
joblib.dump(scaler, "scaler.pkl")

['scaler.pkl']